In [ ]:
# Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")   

In [ ]:
# Load the all dataset

In [ ]:
gold_loans = pd.read_csv('/data/gold_loans.csv')
repayments = pd.read_csv('/data/repayments.csv')
gold_rates = pd.read_csv('/data/gold_rates.csv')
branches = pd.read_csv('/data/branches.csv')
customers = pd.read_csv('/data/customers.csv')

In [ ]:
# Copy all the original dataset

In [ ]:
df_gl = gold_loans.copy()
df_rp = repayments.copy()
df_gr = gold_rates.copy()
df_br = branches.copy()
df_cs = customers.copy()

In [ ]:
# Solve the Questions

In [ ]:
# 1. Size of the loan extract : How many rows are in gold_loans exactly as received (before any cleaning)?


In [ ]:
print('Total Number of Rows in Gold Loans:', df_gl.shape[0])

In [ ]:
# 2. Size of the repayments table : How many rows are in repayments?

In [ ]:
print('Total Number of Rows in Repayments:', df_rp.shape[0])

In [ ]:
# 3. Branch network : How many branches does Suvarna Finance have?


In [ ]:
print('Total Number of Branches in Suvarna Finance:', df_br['branch_id'].nunique())

In [ ]:
# 4. States covered : In how many different states does the company have branches?


In [ ]:
print('Total Number of Different States Covered by Suvarna Finance:', df_br['state'].nunique())

In [ ]:
# 5. Disbursal period : Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal.


In [ ]:
answer = df_gl['disbursal_date'].agg({
    'first_disbursal' : 'min',
    'last_disbursal' : 'max'
}).to_frame(name='disbursal_period').T

answer

In [ ]:
# 6. Unique loans : How many distinct loan_id values are there?


In [ ]:
print('Total Number of Unique Loans:', df_gl['loan_id'].nunique())

In [ ]:
# 7. Duplicate rows : How many exact duplicate rows must be removed from gold_loans?


In [ ]:
print('Total Number of Exact Duplicate Rows in Gold Loans:', df_gl.duplicated().sum())

In [ ]:
# Drop duplicated rows

df_gl.drop_duplicates(inplace=True)

# After Removal of duplicates again check duplicated records
print('Total Duplicate Rows in Gold Loans After Removal:', df_gl.duplicated().sum())

In [ ]:
# 8. Inconsistent status values : In how many different ways is status written in the raw data (count distinct raw values, letter case included)?


In [ ]:
print('Total Number of Distinct Status Values in Raw Data:', df_gl['status'].nunique())

In [ ]:
# 9. Loans by status : After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status.


In [ ]:
# Check Inconsistent Status Values

print('Unique Status Values Before Standardization:', df_gl['status'].unique())

In [ ]:
# Standardize Inconsistent Status Values

df_gl['status'] = df_gl['status'].str.strip().str.lower()

print('Unique Status Values After Standardization:', df_gl['status'].unique())

In [ ]:
answer = df_gl['status'].value_counts().rename('loans')
answer

In [ ]:
# 10. Purity formats : How many different raw values does the purity column contain?


In [ ]:
print('Total Number of Different Raw Values in Purity Column:', df_gl['purity'].nunique())

In [ ]:
# 11. Loans by karat : After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24), return karat and loans for each karat.



In [ ]:
# Check Inconsistent Purity Values

print('Unique Purity Values Before Standardization:', df_gl['purity'].unique())

In [ ]:
# Standardize Inconsistent Purity Values into one scale (karat)

df_gl['purity'] = (
    df_gl['purity']
    .replace(r'\s|K|T', '', regex=True)
    .replace({'916':22,'833':20,'750':18,'999':24})
    .astype(int)
)

print('Unique Status Values After Standardization:', df_gl['purity'].unique())


In [ ]:
answer = df_gl['purity'].value_counts().rename_axis('karat').rename('loans')
answer

In [ ]:
# 12. Missing gold weight : After removing duplicates, how many loans have no gold weight recorded?


In [ ]:
answer = df_gl['gold_weight_gm'].isna().sum()
answer

In [ ]:
# 13. Invalid principal : After removing duplicates, how many loans have a principal of 0 or less?


In [ ]:
answer = len(df_gl[df_gl['principal'] <= 0])
answer

In [ ]:
# 14. Active loans : How many active loans are there after removing duplicates (status 'active' in any letter case)?


In [ ]:
answer = len(df_gl[df_gl['status'] == 'active'])
answer

In [ ]:
# 15. Loans that cannot be assessed : How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?


In [ ]:
answer = len(df_gl[(df_gl['status']=='active') & ((df_gl['principal'] <= 0) | (df_gl['gold_weight_gm'].isnull()))])
answer

In [ ]:
# 16. Gold rate history : How many daily rates are in gold_rates?


In [ ]:
answer = len(df_gr)
answer

In [ ]:
# 17. 2026 peak gold rate : What was the highest 24K gold rate (₹ per gram) during 2026?


In [ ]:
# Convert Datatype of rate_date OBJECT to DATETIME
df_gr['rate_date'] = pd.to_datetime(df_gr['rate_date'], errors='coerce')

answer = df_gr.loc[
    df_gr['rate_date'].dt.year == 2026,
    'rate_24k_per_gram'
].max()

answer

In [ ]:
# 18. Date of the peak : Return one row with peak_date: the date of the highest 24K rate in 2026.


In [ ]:
answer = df_gr[
    (df_gr['rate_date'].dt.year == 2026)
    & 
    (df_gr['rate_24k_per_gram'] == df_gr['rate_24k_per_gram'].max())
]['rate_date'].reset_index(name='peak_date')

answer

In [ ]:
# 19. Rate on the assessment date : Which 24K rate (₹ per gram) applies on 15 October 2026?

In [ ]:
df_gr[df_gr['rate_date'] == '2026-10-15']

In [ ]:
# 20. Fall from the peak : By what percentage has the 24K rate fallen from its 2026 peak to the assessment date? Round to 2 decimals.



In [ ]:
peak_rate = df_gr[df_gr['rate_date'].dt.year == 2026]['rate_24k_per_gram'].max()

assessment_rate = df_gr.loc[df_gr['rate_date'] == '2026-10-15', 'rate_24k_per_gram'].iloc[0]

fall_percentage = round((peak_rate - assessment_rate) / peak_rate * 100, 2)

fall_percentage

In [ ]:
# 21. City names in the branch master : How many distinct city names appear in branches? (Old names such as Bangalore count separately.)

In [ ]:
answer = df_br['city'].nunique()
answer

In [ ]:
# 22. Loans by scheme : After removing duplicates, return scheme and loans for each scheme.

In [ ]:
answer = df_gl.groupby('scheme')['loan_id'].count().reset_index(name='loans')
answer

In [ ]:
# 23. Total disbursed : After removing duplicates, what is the total principal disbursed across all loans with a principal above 0 (₹)?

In [ ]:
df_gl[df_gl['principal'] > 0]['principal'].sum()

In [ ]:
# 24. Average active loan : What is the average principal (₹, rounded to 0 decimals) of assessable active loans?



In [ ]:
round(df_gl[df_gl['status'] == 'active']['principal'].mean(), 0)

In [ ]:
# 25. Customer base : How many customers are in customers?



In [ ]:
len(df_cs)

In [ ]:
# 26. Average customer age : What is the average customer age? Round to 1 decimal.



In [ ]:
df_cs['age'].mean().round(1)

In [ ]:
# 27. Loans disbursed in 2026 : After removing duplicates, how many loans were disbursed in 2026?



In [ ]:
len(df_gl[df_gl['disbursal_date'].dt.year == 2026])

In [ ]:
# 28. Interest rate by scheme : After removing duplicates, return scheme and avg_rate (average interest_rate_pct, rounded to 2 decimals).



In [ ]:
answer = df_gl.groupby('scheme')['interest_rate_pct'].mean().round(2).reset_index(name='avg_rate')
answer

In [ ]:
# 29. Principal repayments : How many repayments have payment_type = 'principal'?



In [ ]:
len(df_rp[df_rp['payment_type'] == 'principal'])

In [ ]:
# 30. Interest collected : What is the total amount received as interest (₹)?



In [ ]:
df_rp.head(5)

In [ ]:
df_rp[df_rp['payment_type'] == 'interest']['amount'].sum()

In [ ]:
# ****************************************** *INTERMEDIATE QUESTIONS* *****************************************

In [ ]:
# 31. Outstanding principal : What is the total outstanding principal (₹) of all assessable active loans? Outstanding = principal − principal repayments.

In [ ]:
# Convert Datatype of rate_date OBJECT to DATETIME
df_gl['disbursal_date'] = pd.to_datetime(df_gl['disbursal_date'], errors='coerce')
df_gl['disbursal_date'].dtypes

In [ ]:
assessment_date = pd.Timestamp('2026-10-15')
rate_asof = df_gr[df_gr['rate_date'] <= assessment_date].sort_values('rate_date')['rate_24k_per_gram'].iloc[-1]

In [ ]:
repaid = df_rp[df_rp['payment_type'] == 'principal'].groupby('loan_id')['amount'].sum()
repaid.head(2)

In [ ]:
book = df_gl[(df_gl['status'] == 'active') & (df_gl['principal'] > 0) & (df_gl['gold_weight_gm'].notna())]
book['outstanding'] = book['principal'] - book['loan_id'].map(repaid).fillna(0)
book['collateral'] = book['gold_weight_gm'] * book['purity'] / 24 * rate_asof
book['current_ltv'] = book['outstanding'] /  book['collateral']

book.head()

In [ ]:
book['outstanding'].sum()

In [ ]:
# 32. Average current LTV : What is the average current LTV (%) of assessable active loans? Round to 2 decimals.

In [ ]:
(100 * book['current_ltv'].mean()).round(2)

In [ ]:
# 33. Loans in breach : How many assessable active loans are above the 75% LTV cap?

In [ ]:
breach = book[book['current_ltv'] > 0.75]
breach.head(3)

In [ ]:
len(book[book['current_ltv'] > 0.75])

In [ ]:
# 34. Money in breach : What is the outstanding principal (₹) of loans above 75% LTV?

In [ ]:
breach['outstanding'].sum()

In [ ]:
# 35. Share of the book in breach : What percentage of assessable active loans are in breach? Round to 2 decimals.

In [ ]:
round(100 * len(breach)/len(book), 2)

In [ ]:
# 36. Breaches by scheme : Return scheme, breach_loans and active_loans (assessable active loans) for each scheme.

In [ ]:
answer = book.groupby('scheme').agg(
    breach_loans = ('current_ltv', lambda x : (x > 0.75).sum()),
    active_loans = ('loan_id', 'count')
).reset_index()

answer

In [ ]:
# 37. Breach exposure by state : Return state and breach_outstanding (₹, rounded to 0) for every state with breaches, highest first.

In [ ]:
answer = breach.merge(df_br, on='branch_id').groupby('state')['outstanding'].sum().round(0).sort_values(ascending=False).reset_index(name='breach_outstanding')
answer

In [ ]:
# 38. Breaches by disbursal month : For breaching loans, return month (text 'YYYY-MM' of the disbursal date) and breach_loans, in month order.

In [ ]:
breach['month'] = breach['disbursal_date'].dt.strftime('%Y-%m')
answer = breach.groupby('month').size().reset_index(name='breach_loans')
answer

In [ ]:
# 39. Top 10 branches : Return the top 10 branches by breached outstanding: branch_id, branch_name, breach_outstanding (₹, rounded to 0), highest first.

In [ ]:
answer = breach.merge(df_br, on='branch_id').groupby(['branch_id', 'branch_name'])['outstanding'].sum().round(0).sort_values(ascending=False).head(10).reset_index(name='breach_outstanding')

answer

In [ ]:
# 40. Express loans in breach : What percentage of assessable active Express loans are in breach? Round to 1 decimal.

In [ ]:
express = book[book['scheme'] == 'Express']

answer = round(100 * (express['current_ltv'] > 0.75).sum() / len(express), 1)
answer

In [ ]:
# 41. Book by region : Return region, active_loans and outstanding (₹, rounded to 0) for assessable active loans in each region.

In [ ]:
active = df_gl[(df_gl['status'] == 'active') & (df_gl['gold_weight_gm'].notna())].copy()

repaid = df_rp[df_rp['payment_type'] == 'principal'].groupby('loan_id')['amount'].sum()

active = active.merge(df_br[['branch_id', 'region']], on='branch_id', how='left')

active['outstanding'] = (active['principal'] - active['loan_id'].map(repaid).fillna(0))


answer = active.groupby('region').agg(
    active_loans=('loan_id', 'count'),
    outstanding=('outstanding', 'sum')
).reset_index().sort_values(by='outstanding', ascending=False)

answer['outstanding'] = answer['outstanding'].round(0)

answer

In [ ]:
print('Active loans with missing gold weight:',
      df_gl[(df_gl['status'] == 'active') & (df_gl['gold_weight_gm'].isna())].shape[0])

print('Zero gold weight:',
      (df_gl['gold_weight_gm'] == 0).sum())

print('Gold weight values:')
print(df_gl['gold_weight_gm'].describe())

print('Purity values:')
print(df_gl['purity'].value_counts(dropna=False))

In [ ]:
df_gl['purity'].value_counts(dropna=False)

In [ ]:
# 42. LTV at disbursal near the peak : For loans disbursed from 1 July to 30 September 2026 (after removing duplicates), return scheme and avg_ltv_at_disbursal (rounded to 2).

In [ ]:
# 43. Overdue active loans : How many active loans (after removing duplicates) are past their maturity date (disbursal date + tenure months) on 15 October 2026?

In [ ]:
# 44. Customers affected : How many distinct customers have at least one loan in breach?

In [ ]:
# 45. Breaches by occupation : Return occupation and breach_loans, highest first.

In [ ]:
breach.head(2)

In [ ]:
# ******************************************** *ADVANCED QUESTIONS* *******************************************

In [ ]:
# 46. A further 5% fall : If the gold rate falls a further 5% from the assessment-date rate, how many assessable active loans would be above 75% LTV?

In [ ]:
# 47. Margin call amount : How much principal (₹, rounded to 0) must be repaid in total to bring every breaching loan back to exactly 75% LTV?

In [ ]:
# 48. Critical breaches : How many assessable active loans are above 80% LTV (critical)?

In [ ]:
# 49. Worst branch in each region : For each region, return the branch with the highest breach rate (% of its assessable active loans in breach): region, branch_name, breach_rate_pct (rounded to 2).

In [ ]:
# 50. Cumulative exposure by month : For breaching loans disbursed in 2026, return month ('YYYY-MM'), breach_outstanding (rounded to 0) and cumulative_outstanding (running total, rounded to 0), in month order.